# Chapter 10: Fourier Transform

## Fourier Transforms

### Python Symbolic and Numerical Computation

In [ ]:
import sympy as sp
from sympy import symbols, exp, integrate, simplify, oo, I, Abs
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# Symbolic variables.  We take omega > 0 to obtain clean closed forms;
# values for omega < 0 follow from Hermitian symmetry, F(-w) = F*(w).
# (SymPy's built-in fourier_transform uses the convention with 2*pi in
# the exponent, so we integrate directly in this chapter's convention.)
t = symbols('t', real=True)
omega, alpha, a = symbols('omega alpha a', positive=True)

def book_ft(f, limits):
    """Fourier transform in this chapter's convention:
    F(omega) = integral of f(t)*exp(-j*omega*t) dt."""
    return simplify(integrate(f * exp(-I * omega * t), limits))

# Example 1: Rectangular pulse (nonzero only on -a <= t <= a)
print("=" * 60)
print("Example 1: Rectangular Pulse")
print("=" * 60)
F_rect = book_ft(1, (t, -a, a))
print("f(t) = rect_a(t)")
print(f"F(omega) = {F_rect}")

# Example 2: Gaussian
print("\n" + "=" * 60)
print("Example 2: Gaussian")
print("=" * 60)
F_gaussian = book_ft(exp(-alpha * t**2), (t, -oo, oo))
print("f(t) = exp(-alpha*t^2)")
print(f"F(omega) = {F_gaussian}")

# Example 3: Two-sided exponential
print("\n" + "=" * 60)
print("Example 3: Two-sided Exponential")
print("=" * 60)
F_exp = book_ft(exp(-alpha * Abs(t)), (t, -oo, oo))
print("f(t) = exp(-alpha*|t|)")
print(f"F(omega) = {F_exp}")

# Example 4: One-sided exponential (the step u(t) restricts t > 0)
print("\n" + "=" * 60)
print("Example 4: One-sided Exponential Decay")
print("=" * 60)
F_exp_causal = book_ft(exp(-alpha * t), (t, 0, oo))
print("f(t) = exp(-alpha*t)*u(t)")
print(f"F(omega) = {F_exp_causal}")

# Numerical example: Gaussian via FFT
print("\n" + "=" * 60)
print("Numerical Example: Gaussian via FFT")
print("=" * 60)
dt = 0.05
t_vals = np.arange(-10, 10, dt)
N = len(t_vals)
alpha_val = 0.5

f_signal = np.exp(-alpha_val * t_vals**2)
F_signal = fft(f_signal) * dt
omega_vals = 2 * np.pi * fftfreq(N, dt)  # angular frequency (rad/s)

# Sort by frequency for plotting
sort_idx = np.argsort(omega_vals)
omega_sorted = omega_vals[sort_idx]
F_sorted = F_signal[sort_idx]

# Analytical transform for comparison
F_exact = np.sqrt(np.pi / alpha_val) * np.exp(-omega_sorted**2
                                              / (4 * alpha_val))
print(f"Max |FFT - analytical|: "
      f"{np.max(np.abs(np.abs(F_sorted) - F_exact)):.3e}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(t_vals, f_signal, 'b-', linewidth=2,
         label='$f(t) = e^{-\\alpha t^2}$')
ax1.set_xlabel('Time $t$ (s)')
ax1.set_ylabel('$f(t)$')
ax1.set_title('Gaussian in Time Domain')
ax1.grid(True)
ax1.legend()

ax2.plot(omega_sorted, np.abs(F_sorted), 'r-', linewidth=2,
         label='$|F(\\omega)|$ (FFT)')
ax2.plot(omega_sorted, F_exact, 'k--', linewidth=1.5, label='Analytical')
ax2.set_xlabel('Angular frequency $\\omega$ (rad/s)')
ax2.set_ylabel('$|F(\\omega)|$')
ax2.set_title('Gaussian in Frequency Domain')
ax2.grid(True)
ax2.legend()
ax2.set_xlim(-8, 8)

plt.tight_layout()
plt.show()

## Properties of Fourier Transforms

### Python Examples of Properties

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq, ifft
from scipy.signal import convolve

# Parameters
dt = 0.01
t = np.arange(-10, 10, dt)
N = len(t)
freqs = fftfreq(N, dt)

# Example 1: Linearity property
print("=" * 60)
print("Property 1: Linearity")
print("=" * 60)

f1 = np.exp(-np.abs(t))
f2 = np.exp(-np.abs(t - 2))
f_linear = 2 * f1 + 3 * f2

F1 = fft(f1) * dt
F2 = fft(f2) * dt
F_linear = fft(f_linear) * dt
F_linear_expected = 2 * F1 + 3 * F2

error = np.max(np.abs(F_linear - F_linear_expected))
print(f"Linearity error: {error:.6e}")

# Example 2: Time shift property
print("\n" + "=" * 60)
print("Property 2: Time Shift")
print("=" * 60)

f_original = np.exp(-np.abs(t))
f_shifted = np.exp(-np.abs(t - 3))

F_original = fft(f_original) * dt
F_shifted = fft(f_shifted) * dt

# Expected phase shift: e^(-j*omega*3)
phase_shift = np.exp(-1j * 3 * 2 * np.pi * freqs)
F_shifted_expected = F_original * phase_shift

error_shift = np.max(np.abs(F_shifted - F_shifted_expected))
print(f"Time-shift property error: {error_shift:.6e}")

# Example 3: Parseval's theorem
print("\n" + "=" * 60)
print("Property 3: Parseval's Theorem")
print("=" * 60)

f = np.exp(-t**2)
F = fft(f) * dt
domega = 2 * np.pi / (N * dt)  # spacing of the frequency grid (rad/s)

energy_time = np.sum(np.abs(f)**2) * dt
energy_freq = np.sum(np.abs(F)**2) * domega / (2 * np.pi)

print(f"Energy (time domain): {energy_time:.6f}")
print(f"Energy (freq domain): {energy_freq:.6f}")
print(f"Parseval error: {abs(energy_time - energy_freq):.6e}")

# Example 4: Convolution theorem
print("\n" + "=" * 60)
print("Property 4: Convolution Theorem")
print("=" * 60)

# Causal signals on [0, 20) keep the FFT's circular convolution
# aligned with the linear one (the tails at t = 20 are negligible).
t_c = np.arange(0, 20, dt)
f_conv1 = np.exp(-t_c)
f_conv2 = np.exp(-2 * t_c)

# Time-domain convolution: O(N^2)
conv_time = convolve(f_conv1, f_conv2)[:len(t_c)] * dt

# Frequency-domain multiplication: O(N log N)
F_conv1 = fft(f_conv1) * dt
F_conv2 = fft(f_conv2) * dt
F_convolution = F_conv1 * F_conv2
conv_freq = ifft(F_convolution) / dt

error_conv = np.max(np.abs(conv_time - np.real(conv_freq)))
print(f"Convolution theorem error: {error_conv:.6e}")

# Example 5: Scaling property
print("\n" + "=" * 60)
print("Property 5: Scaling")
print("=" * 60)

a_scale = 2.0
omega_grid = 2 * np.pi * freqs
# f(t) = exp(-|t|) has the analytic transform F(omega) = 2/(1 + omega^2)
F_analytic = lambda w: 2.0 / (1.0 + w**2)

f_scaled = np.exp(-a_scale * np.abs(t))  # this is f(a*t)
F_scaled = fft(f_scaled) * dt

# Scaling property: F{f(a*t)} = F(omega/a) / |a|
F_scaled_expected = F_analytic(omega_grid / a_scale) / a_scale

error_scaling = np.max(np.abs(np.abs(F_scaled) - F_scaled_expected))
print(f"Scaling property error: {error_scaling:.6e}")

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Plot 1: Linearity
axes[0, 0].plot(t, f1, label='$f_1(t)$')
axes[0, 0].plot(t, f2, label='$f_2(t)$')
axes[0, 0].plot(t, f_linear, label='$2f_1 + 3f_2$', linestyle='--')
axes[0, 0].set_xlabel('Time $t$ (s)')
axes[0, 0].set_title('Linearity Property')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Plot 2: Time shift
axes[0, 1].plot(t, f_original, label='$f(t)$')
axes[0, 1].plot(t, f_shifted, label='$f(t-3)$')
axes[0, 1].set_xlabel('Time $t$ (s)')
axes[0, 1].set_title('Time Shift Property')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Plot 3: Convolution
axes[0, 2].plot(t_c, f_conv1, label='$f_1(t)$')
axes[0, 2].plot(t_c, f_conv2, label='$f_2(t)$')
axes[0, 2].plot(t_c, conv_time/np.max(conv_time),
                label='$f_1 * f_2$ (normalized)')
axes[0, 2].set_xlabel('Time $t$ (s)')
axes[0, 2].set_title('Convolution Property')
axes[0, 2].legend()
axes[0, 2].grid(True)

# Plot magnitude spectra
axes[1, 0].plot(freqs, np.abs(2*F1), label='$|2F_1|$')
axes[1, 0].plot(freqs, np.abs(3*F2), label='$|3F_2|$')
axes[1, 0].plot(freqs, np.abs(F_linear), label='$|2F_1 + 3F_2|$',
                linestyle='--')
axes[1, 0].set_xlim(-2, 2)
axes[1, 0].set_xlabel('Frequency $f$ (Hz)')
axes[1, 0].set_title('Linearity in Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(freqs, np.abs(F_original), label='$|F(\\omega)|$')
axes[1, 1].plot(freqs, np.abs(F_shifted),
                label='$|e^{-3j\\omega} F(\\omega)|$', linestyle='--')
axes[1, 1].set_xlim(-2, 2)
axes[1, 1].set_xlabel('Frequency $f$ (Hz)')
axes[1, 1].set_title('Time Shift in Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True)

axes[1, 2].semilogy(freqs[:len(freqs)//2],
                    np.abs(F_conv1[:len(freqs)//2]), label='$|F_1|$')
axes[1, 2].semilogy(freqs[:len(freqs)//2],
                    np.abs(F_conv2[:len(freqs)//2]), label='$|F_2|$')
axes[1, 2].semilogy(freqs[:len(freqs)//2],
                    np.abs(F_convolution[:len(freqs)//2]),
                    label='$|F_1 \\cdot F_2|$', linestyle='--')
axes[1, 2].set_xlim(0, 2)
axes[1, 2].set_xlabel('Frequency $f$ (Hz)')
axes[1, 2].set_title('Convolution in Frequency')
axes[1, 2].legend()
axes[1, 2].grid(True)

plt.tight_layout()
plt.show()

## Delta Function

### SymPy and NumPy Delta Function

In [ ]:
import sympy as sp
from sympy import symbols, DiracDelta, diff, integrate, oo, fourier_transform
import numpy as np
import matplotlib.pyplot as plt

# Symbolic computation
print("=" * 60)
print("Dirac Delta Function in SymPy")
print("=" * 60)

t, t0, omega = symbols('t t0 omega', real=True)

# Define delta functions
delta = DiracDelta(t)
delta_shifted = DiracDelta(t - t0)

# Sifting property
print("\n1. Sifting Property:")
f = t**2 + 3*t + 1
result = integrate(f * delta_shifted, (t, -oo, oo))
expected = f.subs(t, t0)
print(f"   Integral of (t^2 + 3t + 1) * delta(t - t0): {result}")
print(f"   Expected (f(t0)): {expected}")
print(f"   Match: {result == expected}")

# Properties
print("\n2. Basic Properties:")
delta_neg = DiracDelta(-t)
print(f"   delta(-t) = {delta_neg} = {delta}")
integral_delta = integrate(delta, (t, -oo, oo))
print(f"   Integral of delta(t): {integral_delta}")

# Derivative of delta
print("\n3. Derivative of Delta:")
delta_prime = diff(delta, t)
print(f"   d(delta(t))/dt = {delta_prime}")

# Sifting with derivative
f_deriv = diff(f, t)
result_deriv = integrate(f * delta_prime, (t, -oo, oo))
expected_deriv = -f_deriv.subs(t, 0)
print(f"   Integral of f(t) * delta'(t): {result_deriv}")
print(f"   Expected (-f'(0)): {expected_deriv}")

# Fourier transform of delta.  SymPy's fourier_transform uses the
# convention F(nu) = Integral f(t) exp(-2*pi*I*nu*t) dt; substituting
# nu -> omega/(2*pi) converts to this chapter's angular-frequency form.
print("\n4. Fourier Transform of Delta:")
F_delta = fourier_transform(delta, t, omega)
print(f"   F{{delta(t)}} = {F_delta}")

F_delta_shifted = fourier_transform(delta_shifted, t, omega)
F_delta_shifted = F_delta_shifted.subs(omega, omega / (2 * sp.pi))
print(f"   F{{delta(t - t0)}} = {F_delta_shifted}")

# Fourier transform of constant (using duality)
print("\n5. Fourier Transform of Constants:")
print(f"   F{{1}} = 2*pi*delta(omega)")
print(f"   F{{delta(t)}} = 1")

# Numerical approximation of delta function
print("\n" + "=" * 60)
print("Numerical Delta Function Approximation")
print("=" * 60)

t_vals = np.linspace(-5, 5, 1000)

# Gaussian approximation
epsilon = 0.2
delta_gaussian = (1 / np.sqrt(np.pi * epsilon)) * np.exp(-t_vals**2 / epsilon)

# Rectangular approximation
delta_rect = np.where(np.abs(t_vals) <= epsilon/2, 1/epsilon, 0)

# Sinc approximation: sin(t/eps)/(pi*t).
# NumPy's np.sinc(x) is sin(pi*x)/(pi*x), hence the rescaling below.
delta_sinc = np.sinc(t_vals / (np.pi * epsilon)) / (np.pi * epsilon)

# Test sifting property with f(t) = cos(t); expected value f(0) = 1
f_vals = np.cos(t_vals)
sifting_gaussian = np.trapezoid(f_vals * delta_gaussian, t_vals)
sifting_rect = np.trapezoid(f_vals * delta_rect, t_vals)
sifting_sinc = np.trapezoid(f_vals * delta_sinc, t_vals)

print(f"\nSifting property: integral of cos(t)*delta_eps(t)")
print(f"Gaussian approx (epsilon={epsilon}): {sifting_gaussian:.6f}")
print(f"Rectangular approx (epsilon={epsilon}): {sifting_rect:.6f}")
print(f"Sinc approx (epsilon={epsilon}): {sifting_sinc:.6f}")
print(f"Expected (f(0) = cos(0)): 1.0")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot 1: Gaussian approximation
axes[0, 0].plot(t_vals, delta_gaussian, 'b-', linewidth=2)
axes[0, 0].fill_between(t_vals, 0, delta_gaussian, alpha=0.3)
axes[0, 0].set_title(f'Gaussian Approximation ($\\epsilon = {epsilon}$)')
axes[0, 0].set_xlim(-2, 2)
axes[0, 0].grid(True)
axes[0, 0].set_ylabel('$\\delta_\\epsilon(t)$')

# Plot 2: Rectangular approximation
axes[0, 1].plot(t_vals, delta_rect, 'r-', linewidth=2)
axes[0, 1].fill_between(t_vals, 0, delta_rect, alpha=0.3)
axes[0, 1].set_title(f'Rectangular Approximation ($\\epsilon = {epsilon}$)')
axes[0, 1].set_xlim(-2, 2)
axes[0, 1].grid(True)

# Plot 3: Sinc approximation
axes[1, 0].plot(t_vals, delta_sinc, 'g-', linewidth=2)
axes[1, 0].fill_between(t_vals, 0, delta_sinc, alpha=0.3, where=(delta_sinc >= 0))
axes[1, 0].set_title(f'Sinc Approximation ($\\epsilon = {epsilon}$)')
axes[1, 0].set_xlim(-2, 2)
axes[1, 0].grid(True)
axes[1, 0].set_xlabel('$t$')
axes[1, 0].set_ylabel('$\\delta_\\epsilon(t)$')

# Plot 4: Comparison of convergence
epsilons = [1.0, 0.5, 0.2, 0.1]
colors = ['blue', 'red', 'green', 'purple']

for eps, color in zip(epsilons, colors):
    delta_gauss_eps = (1 / np.sqrt(np.pi * eps)) * np.exp(-t_vals**2 / eps)
    axes[1, 1].plot(t_vals, delta_gauss_eps, color=color, linewidth=2, label=f'$\\epsilon = {eps}$')

axes[1, 1].set_title('Gaussian Approximations Converging to $\\delta(t)$')
axes[1, 1].set_xlim(-1.5, 1.5)
axes[1, 1].legend()
axes[1, 1].grid(True)
axes[1, 1].set_xlabel('$t$')

plt.tight_layout()
plt.show()

## Inverse Fourier Transform

### Python Examples of Inverse Transform

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import ifft, fft, fftfreq
from scipy.signal import residue

# Symbolic inverse transforms
print("=" * 60)
print("Symbolic Inverse Fourier Transforms (SymPy)")
print("=" * 60)

omega = sp.symbols('omega', real=True)
t = sp.symbols('t', real=True, positive=True)  # t > 0: causal signals

def book_ift(F):
    """Inverse transform in this chapter's convention,
    f(t) = (1/(2*pi)) * Integral F(omega) exp(j*omega*t) domega.
    SymPy's inverse_fourier_transform places 2*pi in the exponent,
    so we rescale omega before calling it."""
    return sp.inverse_fourier_transform(
        F.subs(omega, 2 * sp.pi * omega), omega, t)

# Example 1: Simple rational function
print("\nExample 1: F(omega) = 1/(1 + j*omega)")
F1 = 1 / (1 + sp.I * omega)
f1 = book_ift(F1)
print(f"f(t) = {f1}  (for t > 0)")

# Example 2: Double pole -- verify the pair in the forward direction
print("\nExample 2: f(t) = t*exp(-t)*u(t)  <->  1/(1 + j*omega)^2")
w = sp.symbols('w', positive=True)
F2 = sp.simplify(sp.integrate(t * sp.exp(-t) * sp.exp(-sp.I * w * t),
                              (t, 0, sp.oo)))
print(f"F(w) = {F2}")

# Example 3: Product of first-order terms (partial fractions first)
print("\nExample 3: F(omega) = 1/((1 + j*omega)(2 + j*omega))")
F3 = 1 / ((1 + sp.I * omega) * (2 + sp.I * omega))
f3 = sp.simplify(book_ift(sp.apart(F3, omega)))
print(f"f(t) = {f3}  (for t > 0)")

# Example 4: Gaussian in frequency domain
print("\nExample 4: F(omega) = exp(-omega^2)")
F4 = sp.exp(-omega**2)
f4 = book_ift(F4)
print(f"f(t) = {f4}")

# Partial fractions method
print("\n" + "=" * 60)
print("Partial Fractions Method")
print("=" * 60)

# Transfer function: 1 / (s^2 + 3s + 2) = 1 / ((s+1)(s+2))
# Numerator and denominator coefficients
num = [1]
den = [1, 3, 2]

# Compute partial fractions
residues, poles, const = residue(num, den)

print(f"\nTransfer function: 1 / (s^2 + 3s + 2)")
print(f"Poles: {poles}")
print(f"Residues: {residues}")
print(f"Partial fractions:")
for r, p in zip(residues, poles):
    print(f"  {r} / (s - ({p}))")

# Time domain: each term 1/(s - p) becomes exp(p*t)*u(t)
print(f"\nTime domain impulse response:")
print(f"  h(t) = {residues[0]} * exp({poles[0]}*t)*u(t) + {residues[1]} * exp({poles[1]}*t)*u(t)")

# Numerical IFFT
print("\n" + "=" * 60)
print("Numerical Inverse Transform via IFFT")
print("=" * 60)

# Create a time signal
dt = 0.01
t_vals = np.arange(0, 5, dt)
N = len(t_vals)

# Frequency domain signal (one-sided exponential)
# f(t) = exp(-t)*u(t) <-> F(omega) = 1/(1 + j*omega)
f_true = np.exp(-t_vals)

# Forward FFT
F_signal = fft(f_true) * dt
freqs = fftfreq(N, dt)

# Inverse FFT
f_recovered = ifft(F_signal) / dt

# Reconstruction error
error = np.max(np.abs(f_true - np.real(f_recovered)))
print(f"\nSignal: f(t) = exp(-t)*u(t)")
print(f"Maximum reconstruction error (IFFT): {error:.6e}")

# Example with Gaussian
print("\n" + "=" * 60)
print("Gaussian Signal Example")
print("=" * 60)

t_vals_gaussian = np.arange(-5, 5, 0.05)
N_gaussian = len(t_vals_gaussian)

# Create Gaussian signal
f_gaussian = np.exp(-0.5 * t_vals_gaussian**2)

# Forward and inverse
F_gaussian = fft(f_gaussian) * 0.05
f_gaussian_recovered = ifft(F_gaussian) / 0.05

error_gaussian = np.max(np.abs(f_gaussian - np.real(f_gaussian_recovered)))
print(f"Signal: f(t) = exp(-0.5*t^2)")
print(f"Maximum reconstruction error (IFFT): {error_gaussian:.6e}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot 1: Exponential signal and recovery
axes[0, 0].plot(t_vals, f_true, 'b-', linewidth=2, label='Original')
axes[0, 0].plot(t_vals, np.real(f_recovered), 'r--', linewidth=1.5,
                label='Recovered')
axes[0, 0].set_xlabel('Time $t$ (s)')
axes[0, 0].set_ylabel('$f(t)$')
axes[0, 0].set_title('Exponential Signal Recovery')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Plot 2: Frequency domain
axes[0, 1].plot(freqs, np.abs(F_signal), 'g-', linewidth=2)
axes[0, 1].set_xlabel('Frequency $f$ (Hz)')
axes[0, 1].set_ylabel('$|F|$')
axes[0, 1].set_title('Frequency Spectrum of $f(t) = e^{-t}u(t)$')
axes[0, 1].set_xlim(-5, 5)
axes[0, 1].grid(True)

# Plot 3: Gaussian signal
axes[1, 0].plot(t_vals_gaussian, f_gaussian, 'b-', linewidth=2,
                label='Original')
axes[1, 0].plot(t_vals_gaussian, np.real(f_gaussian_recovered), 'r--',
                linewidth=1.5, label='Recovered')
axes[1, 0].set_xlabel('Time $t$ (s)')
axes[1, 0].set_ylabel('$f(t)$')
axes[1, 0].set_title('Gaussian Signal Recovery')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Plot 4: Reconstruction error
freqs_gaussian = fftfreq(N_gaussian, 0.05)
error_spectrum = np.abs(fft(f_gaussian) * 0.05
                        - fft(f_gaussian_recovered) * 0.05)
axes[1, 1].semilogy(freqs_gaussian, error_spectrum, 'k-', linewidth=2)
axes[1, 1].set_xlabel('Frequency $f$ (Hz)')
axes[1, 1].set_ylabel('Reconstruction error')
axes[1, 1].set_title('Reconstruction Error Spectrum')
axes[1, 1].set_xlim(-5, 5)
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## Solution of ODEs with Fourier Transform

### Python Verification of ODE Solutions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.fft import fft, ifft, fftfreq

# Example 1: Damped harmonic oscillator
print("=" * 60)
print("Example 1: Damped Harmonic Oscillator")
print("=" * 60)

# Parameters
omega_0 = 1.0  # Natural frequency
gamma = 0.3    # Damping coefficient
dt = 0.01
t = np.arange(0, 20, dt)

# Forcing function: impulse followed by exponential decay
f_t = np.exp(-0.5 * t)

# Method 1: Time-domain convolution (impulse response)
h_t = (1 / np.sqrt(omega_0**2 - gamma**2)) * np.exp(-gamma * t) * \
      np.sin(np.sqrt(omega_0**2 - gamma**2) * t)

# Causal convolution: keep the first len(t) samples of the full result
y_conv = np.convolve(h_t, f_t)[:len(t)] * dt

# Method 2: Frequency-domain multiplication
F_f = fft(f_t) * dt
freqs = fftfreq(len(t), dt)
omega = 2 * np.pi * freqs

# Transfer function
H = 1 / (omega_0**2 - omega**2 + 2j * gamma * omega)
F_y = H * F_f
y_freq = np.real(ifft(F_y)) / dt

# Method 3: ODE integration using odeint
def damped_osc(y_state, t, omega_0, gamma, f_func):
    y, dydt = y_state
    d2ydt2 = f_func(t) - 2*gamma*dydt - omega_0**2*y
    return [dydt, d2ydt2]

def f_forcing(t):
    return np.exp(-0.5 * t)

y0 = [0, 0]
y_odeint = odeint(damped_osc, y0, t, args=(omega_0, gamma, f_forcing))

print(f"Omega_0 = {omega_0}, Gamma = {gamma}")
print(f"Max difference (conv vs freq): {np.max(np.abs(y_conv - y_freq)):.6e}")
print(f"Max difference (conv vs odeint): {np.max(np.abs(y_conv - y_odeint[:, 0])):.6e}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Forcing function
axes[0, 0].plot(t, f_t, 'k-', linewidth=2)
axes[0, 0].set_xlabel('Time $t$ (s)')
axes[0, 0].set_ylabel('$f(t)$')
axes[0, 0].set_title('Forcing Function')
axes[0, 0].grid(True)

# Impulse response
axes[0, 1].plot(t, h_t, 'b-', linewidth=2)
axes[0, 1].set_xlabel('Time $t$ (s)')
axes[0, 1].set_ylabel('$h(t)$')
axes[0, 1].set_title('Impulse Response')
axes[0, 1].grid(True)

# Solutions comparison
axes[1, 0].plot(t, y_conv, 'b-', linewidth=2, label='Convolution')
axes[1, 0].plot(t, y_odeint[:, 0], 'r--', linewidth=1.5,
                label='ODE integration')
axes[1, 0].set_xlabel('Time $t$ (s)')
axes[1, 0].set_ylabel('$y(t)$')
axes[1, 0].set_title('Solution: Damped Harmonic Oscillator')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Transfer function magnitude
H_magnitude = np.abs(H)
valid_idx = np.abs(freqs) < 3
axes[1, 1].semilogy(freqs[valid_idx], H_magnitude[valid_idx], 'g-',
                    linewidth=2)
axes[1, 1].set_xlabel('Frequency $f$ (Hz)')
axes[1, 1].set_ylabel('$|H|$')
axes[1, 1].set_title('Transfer Function Magnitude')
axes[1, 1].grid(True, which='both')

plt.tight_layout()
plt.show()

# Example 2: Beam deflection
print("\n" + "=" * 60)
print("Example 2: Beam Deflection")
print("=" * 60)

x = np.linspace(0, 10, 200)
dx = x[1] - x[0]

# Distributed load: triangular
q_x = np.maximum(1 - np.abs(x - 5) / 2.5, 0)

# Beam deflection: y = (1/6EI) * integral of (x - xi)^3 * q(xi)
EI = 1000.0
y_beam = np.zeros_like(x)
for i, x_i in enumerate(x):
    y_beam[i] = (1 / (6 * EI)) * np.trapezoid(
        (x_i - x[:i+1])**3 * q_x[:i+1], x[:i+1])

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x, q_x, 'k-', linewidth=2)
axes[0].fill_between(x, 0, q_x, alpha=0.3)
axes[0].set_xlabel('Position $x$ (m)')
axes[0].set_ylabel('Load $q(x)$ (N/m)')
axes[0].set_title('Distributed Load')
axes[0].grid(True)

axes[1].plot(x, y_beam, 'b-', linewidth=2)
axes[1].set_xlabel('Position $x$ (m)')
axes[1].set_ylabel('Deflection $y(x)$ (m)')
axes[1].set_title('Beam Deflection')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Heat Equation on Infinite Domain

### Python Numerical Solution and Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc

# Example 1: Heat equation with Gaussian initial condition
print("=" * 60)
print("Heat Equation: Gaussian Initial Condition")
print("=" * 60)

# Parameters
alpha = 0.5
L = 10
nx = 200
nt = 100
x = np.linspace(-L, L, nx)
t = np.linspace(0, 5, nt)
dx = x[1] - x[0]
dt = t[1] - t[0]

# Initial condition: Gaussian
u_init = np.exp(-x**2 / 2)

# Analytical solution via Gaussian kernel
def heat_solution_gaussian(x, t, alpha, u_init_func, dx_domain=np.linspace(-10, 10, 500)):
    """Compute heat equation solution via convolution with Gaussian kernel"""
    if t == 0:
        return u_init_func(x)

    kernel = (1 / np.sqrt(4 * np.pi * alpha * t)) * \
             np.exp(-(x[:, np.newaxis] - dx_domain)**2 / (4 * alpha * t))
    u_vals = u_init_func(dx_domain)
    u_solution = np.trapezoid(kernel * u_vals, dx_domain, axis=1)
    return u_solution

# Compute solution
u_solution = np.zeros((nt, nx))
for n, t_n in enumerate(t):
    u_solution[n, :] = heat_solution_gaussian(x, t_n, alpha,
                                               lambda x: np.exp(-x**2 / 2),
                                               dx_domain=x)

# Check against the exact self-similar solution: a Gaussian initial
# condition of variance 1 stays Gaussian with variance 1 + 2*alpha*t
sigma2 = 1 + 2 * alpha * t[-1]
u_exact = np.exp(-x**2 / (2 * sigma2)) / np.sqrt(sigma2)
interior = np.abs(x) < 5  # away from the truncated domain edges
err = np.max(np.abs(u_solution[-1, interior] - u_exact[interior]))
print(f"Max |numerical - exact| at t = {t[-1]:.1f}: {err:.3e}")

# Visualization
fig = plt.figure(figsize=(12, 5))

# Line plots at different times
ax1 = fig.add_subplot(121)
for n in [0, 25, 50, 75, 99]:
    ax1.plot(x, u_solution[n, :], linewidth=2, label=f't = {t[n]:.2f} s')
ax1.set_xlabel('Position $x$ (m)')
ax1.set_ylabel('Temperature $u(x,t)$')
ax1.set_title('Heat Diffusion: Temperature Profiles')
ax1.legend()
ax1.grid(True)

# Contour plot
ax2 = fig.add_subplot(122)
X, T = np.meshgrid(x, t)
contour = ax2.contourf(X, T, u_solution, levels=20, cmap='hot')
ax2.set_xlabel('Position $x$ (m)')
ax2.set_ylabel('Time $t$ (s)')
ax2.set_title('Heat Equation: Space-Time Evolution')
cbar = plt.colorbar(contour, ax=ax2)
cbar.set_label('Temperature $u(x,t)$')

plt.tight_layout()
plt.show()

# Example 2: Semi-infinite domain with Dirichlet boundary condition
print("\n" + "=" * 60)
print("Semi-infinite Domain: Dirichlet Boundary Condition")
print("=" * 60)

x_semi = np.linspace(0, 10, 150)
t_semi = np.array([0.1, 0.5, 1.0, 2.0, 5.0])

fig, ax = plt.subplots(figsize=(10, 5))

for t_val in t_semi:
    u_semi = erfc(x_semi / (2 * np.sqrt(alpha * t_val)))
    ax.plot(x_semi, u_semi, linewidth=2.5, label=f't = {t_val} s')

ax.set_xlabel('Position $x$ (m)')
ax.set_ylabel('Temperature $u(x,t)$')
ax.set_title('Heat Equation: Semi-infinite Domain (u(0,t) = 1)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Laplace's Equation via Fourier Transform

### Python Computation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# Parameters
x_domain = np.linspace(-5, 5, 200)
y_domain = np.linspace(0.1, 5, 100)

# Boundary condition: rectangular pulse
def f_boundary(xi):
    return 1.0 if np.abs(xi) < 1.0 else 0.0

# Poisson kernel
def poisson_kernel(x, y, xi):
    return y / (np.pi * ((x - xi)**2 + y**2))

# Solution via Poisson integral.  The boundary data vanish outside
# [-1, 1], so the integration can be restricted to that interval.
u_solution = np.zeros((len(y_domain), len(x_domain)))

for j, y in enumerate(y_domain):
    for i, x in enumerate(x_domain):
        integrand = lambda xi: poisson_kernel(x, y, xi) * f_boundary(xi)
        u_solution[j, i], _ = quad(integrand, -1, 1)

# Check against the closed-form arctan solution of the example
X, Y = np.meshgrid(x_domain, y_domain)
u_exact = (np.arctan((1 - X) / Y) - np.arctan((-1 - X) / Y)) / np.pi
print(f"Max |quadrature - closed form|: "
      f"{np.max(np.abs(u_solution - u_exact)):.3e}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contour plot
contour = axes[0].contourf(X, Y, u_solution, levels=20, cmap='viridis')
axes[0].set_xlabel('Position $x$ (m)')
axes[0].set_ylabel('Height $y$ (m)')
axes[0].set_title('Solution to Laplace Equation (Half-plane)')
cbar = plt.colorbar(contour, ax=axes[0])
cbar.set_label('$u(x,y)$')

# Boundary and cross-sections
axes[1].fill_between([-1, 1], 0, 1, alpha=0.3, label='Boundary condition')
for y_val in [0.5, 1.0, 2.0]:
    idx = np.argmin(np.abs(y_domain - y_val))
    axes[1].plot(x_domain, u_solution[idx, :], linewidth=2,
                 label=f'y = {y_val}')
axes[1].set_xlabel('Position $x$ (m)')
axes[1].set_ylabel('$u(x,y)$')
axes[1].set_title('Cross-sections of Solution')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Wave Equation

### Wave Packet Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
c = 1.0  # Wave speed
x = np.linspace(-10, 10, 400)
t_vals = np.array([0, 1, 2, 3, 4])

# Initial displacement: Gaussian packet
def initial_displacement(x):
    return np.exp(-x**2 / 2)

# Initial velocity: zero
def initial_velocity(x):
    return np.zeros_like(x)

# D'Alembert solution
def d_alembert(x, t, f, g, c):
    """Compute wave solution using d'Alembert formula"""
    u_right = np.interp(x + c*t, x, f, left=0, right=0)
    u_left = np.interp(x - c*t, x, f, left=0, right=0)

    # Integral term for velocity
    integral = np.zeros_like(x)
    if np.max(np.abs(g)) > 0:
        for i in range(len(x)):
            x_int = np.linspace(max(x[0], x[i] - c*t), min(x[-1], x[i] + c*t), 100)
            g_int = np.interp(x_int, x, g, left=0, right=0)
            integral[i] = np.trapezoid(g_int, x_int)

    return 0.5 * (u_right + u_left) + 0.5 / c * integral

# Compute solution at different times
u_solution = np.zeros((len(t_vals), len(x)))
f = initial_displacement(x)
g = initial_velocity(x)

for n, t in enumerate(t_vals):
    u_solution[n, :] = d_alembert(x, t, f, g, c)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Line plots: wave propagation
for n, t in enumerate(t_vals):
    axes[0].plot(x, u_solution[n, :], linewidth=2, label=f't = {t} s')
axes[0].set_xlabel('Position $x$ (m)')
axes[0].set_ylabel('$u(x,t)$')
axes[0].set_title('Wave Packet Propagation (D\'Alembert Solution)')
axes[0].legend()
axes[0].grid(True)
axes[0].set_ylim(-0.2, 1.1)

# Space-time diagram
T, X = np.meshgrid(t_vals, x)
contour = axes[1].contourf(T, X, u_solution.T, levels=20, cmap='RdBu_r')
axes[1].set_xlabel('Time $t$ (s)')
axes[1].set_ylabel('Position $x$ (m)')
axes[1].set_title('Wave Equation: Space-Time Diagram')
cbar = plt.colorbar(contour, ax=axes[1])
cbar.set_label('u(x,t)')

# Characteristic lines
for x0 in [-8, -4, 0, 4, 8]:
    axes[1].plot(t_vals, x0 + c*t_vals, 'k--', linewidth=0.5, alpha=0.5)
    axes[1].plot(t_vals, x0 - c*t_vals, 'k--', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.show()

## Applications to Signal and Image Processing

### Python Examples: Filtering and Spectral Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq

np.random.seed(42)  # reproducible noise

# Example 1: Low-pass and high-pass filtering
print("=" * 60)
print("Signal Filtering Example")
print("=" * 60)

# Create a composite signal
dt = 0.01
t = np.arange(0, 10, dt)
N = len(t)

# Low frequency component (1 Hz)
signal_low = 2 * np.sin(2 * np.pi * 1 * t)

# High frequency component (8 Hz)
signal_high = 0.5 * np.sin(2 * np.pi * 8 * t)

# Composite signal
signal = signal_low + signal_high + 0.2 * np.random.randn(N)

# FFT
freqs = fftfreq(N, dt)
F_signal = fft(signal) * dt

# Design ideal low-pass filter
cutoff_freq = 3.0  # Hz
H_lpf = np.where(np.abs(freqs) < cutoff_freq, 1, 0)

# Apply filter
F_filtered_lpf = H_lpf * F_signal
signal_filtered_lpf = np.real(ifft(F_filtered_lpf)) / dt

# High-pass filter
H_hpf = 1 - H_lpf
F_filtered_hpf = H_hpf * F_signal
signal_filtered_hpf = np.real(ifft(F_filtered_hpf)) / dt

# Visualization
fig, axes = plt.subplots(3, 2, figsize=(12, 10))

# Original signal
axes[0, 0].plot(t, signal, 'k-', linewidth=1, alpha=0.7)
axes[0, 0].set_xlabel('Time $t$ (s)')
axes[0, 0].set_ylabel('Signal')
axes[0, 0].set_title('Original Signal (Noisy)')
axes[0, 0].grid(True)

# Frequency spectrum
valid_idx = np.abs(freqs) < 15
axes[0, 1].semilogy(freqs[valid_idx], np.abs(F_signal[valid_idx]), 'k-', linewidth=1.5)
axes[0, 1].set_ylabel('$|F(\\omega)|$')
axes[0, 1].set_title('Frequency Spectrum')
axes[0, 1].grid(True, which='both')

# Low-pass filtered signal
axes[1, 0].plot(t, signal_low, 'b--', linewidth=1.5, label='Ground truth (low freq)')
axes[1, 0].plot(t, signal_filtered_lpf, 'r-', linewidth=1.5, label='Filtered')
axes[1, 0].set_ylabel('Signal')
axes[1, 0].set_title('Low-pass Filtered (cutoff = 3 Hz)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Low-pass filter response
axes[1, 1].fill_between(freqs[valid_idx], 0, H_lpf[valid_idx], alpha=0.5)
axes[1, 1].set_ylabel('$H(\\omega)$')
axes[1, 1].set_title('Low-pass Filter Response')
axes[1, 1].grid(True)
axes[1, 1].set_xlim(-15, 15)

# High-pass filtered signal
axes[2, 0].plot(t, signal_high, 'b--', linewidth=1.5, label='Ground truth (high freq)')
axes[2, 0].plot(t, signal_filtered_hpf, 'r-', linewidth=1.5, label='Filtered')
axes[2, 0].set_xlabel('Time $t$ (s)')
axes[2, 0].set_ylabel('Signal')
axes[2, 0].set_title('High-pass Filtered')
axes[2, 0].legend()
axes[2, 0].grid(True)

# High-pass filter response
axes[2, 1].fill_between(freqs[valid_idx], 0, H_hpf[valid_idx], alpha=0.5)
axes[2, 1].set_xlabel('Frequency (Hz)')
axes[2, 1].set_ylabel('$H(\\omega)$')
axes[2, 1].set_title('High-pass Filter Response')
axes[2, 1].grid(True)
axes[2, 1].set_xlim(-15, 15)

plt.tight_layout()
plt.show()

# Example 2: 2D FFT for image processing
print("\n" + "=" * 60)
print("2D FFT: Image Processing")
print("=" * 60)

# Create a simple synthetic image (pattern)
nx, ny = 128, 128
x_2d = np.linspace(-5, 5, nx)
y_2d = np.linspace(-5, 5, ny)
X, Y = np.meshgrid(x_2d, y_2d)

# Image: sum of Gaussians
image = np.exp(-((X-1)**2 + (Y-1)**2) / 0.5) + 0.5*np.exp(-((X+1)**2 + (Y+1)**2) / 1.0)
image += 0.1 * np.random.randn(nx, ny)

# 2D FFT
F_image = fft(fft(image, axis=0), axis=1)
power_spectrum = np.abs(F_image)**2

# Low-pass filter in 2D
freqs_x = np.fft.fftfreq(nx, x_2d[1] - x_2d[0])
freqs_y = np.fft.fftfreq(ny, y_2d[1] - y_2d[0])
FX, FY = np.meshgrid(freqs_x, freqs_y)
radius = np.sqrt(FX**2 + FY**2)

cutoff_2d = 2.0
H_2d = np.where(radius < cutoff_2d, 1, 0)
F_filtered_2d = H_2d * F_image
image_filtered = np.real(ifft(ifft(F_filtered_2d, axis=1), axis=0))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original image
axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original Image')

# Power spectrum (log scale)
axes[0, 1].imshow(np.log(1 + power_spectrum), cmap='hot')
axes[0, 1].set_title('Power Spectrum (log scale)')

# Filter mask
axes[1, 0].imshow(H_2d, cmap='gray')
axes[1, 0].set_title(f'2D Low-pass Filter Mask (cutoff = {cutoff_2d})')

# Filtered image
axes[1, 1].imshow(image_filtered, cmap='gray')
axes[1, 1].set_title('Low-pass Filtered Image')

plt.tight_layout()
plt.show()

## Wavelet Transform

### Python Implementation with PyWavelets

In [ ]:
import numpy as np
import pywt
import matplotlib.pyplot as plt

np.random.seed(42)  # reproducible noise

# Create a test signal with multiple frequency components
t = np.linspace(0, 10, 1000)
# High frequency component (0-2 seconds)
signal_part1 = 2 * np.sin(2 * np.pi * 5 * t[:200])
# Low frequency component (2-10 seconds)
signal_part2 = 3 * np.sin(2 * np.pi * 1 * t[200:])
test_signal = np.concatenate([signal_part1, signal_part2])
# Add noise
test_signal += 0.5 * np.random.randn(len(t))

# Continuous Wavelet Transform.  pywt.cwt returns the coefficient
# matrix together with the frequencies that correspond to each scale.
scales = np.arange(1, 128)
coefficients_cwt, freqs_cwt = pywt.cwt(test_signal, scales, 'morl',
                                       sampling_period=t[1] - t[0])

# Plot CWT
fig, axes = plt.subplots(3, 1, figsize=(13, 10))

# Original signal
ax = axes[0]
ax.plot(t, test_signal, 'b-', linewidth=1.5)
ax.axvline(x=2, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Amplitude', fontsize=11)
ax.set_title('Original Signal (High Freq 0-2s, Low Freq 2-10s)',
             fontsize=12)
ax.grid(True, alpha=0.3)

# CWT
ax = axes[1]
im = ax.contourf(t, scales, np.abs(coefficients_cwt), levels=64,
                  cmap='viridis')
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Scale', fontsize=11)
ax.set_title('Continuous Wavelet Transform (Morlet Wavelet)',
             fontsize=12)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Magnitude', fontsize=10)

# Discrete Wavelet Transform
coeffs = pywt.wavedec(test_signal, 'db4', level=5)
cA, cD5, cD4, cD3, cD2, cD1 = coeffs

# Plot DWT
ax = axes[2]
offset = 0
lengths = [len(cA), len(cD5), len(cD4), len(cD3), len(cD2), len(cD1)]
labels = ['cA5', 'cD5', 'cD4', 'cD3', 'cD2', 'cD1']
colors = ['b', 'r', 'g', 'c', 'm', 'y']  # single-letter format codes

for i, (coeff, length, label, color) in enumerate(zip(
    [cA, cD5, cD4, cD3, cD2, cD1], lengths, labels, colors)):
    x_pos = np.arange(offset, offset + length)
    ax.stem(x_pos, coeff, linefmt=f'{color}-', markerfmt=f'{color}o',
            basefmt=' ', label=label)
    offset += length

ax.set_xlabel('Coefficient index', fontsize=11)
ax.set_ylabel('Coefficient Value', fontsize=11)
ax.legend(fontsize=9, ncol=6, loc='upper right')
ax.set_title('Discrete Wavelet Transform (Daubechies db4)',
             fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('wavelet_transform.png', dpi=200)
plt.close()

# Signal denoising using wavelets
def denoise_signal(signal, wavelet, level, threshold_type='soft'):
    # Decompose
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    cA = coeffs[0]
    cD_list = coeffs[1:]

    # Apply thresholding to detail coefficients
    sigma = np.median(np.abs(cD_list[-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(len(signal)))

    cD_thresholded = [
        pywt.threshold(cD, threshold, mode=threshold_type)
        for cD in cD_list
    ]

    # Reconstruct
    denoised = pywt.waverec([cA] + cD_thresholded, wavelet)
    return denoised[:len(signal)]

# Denoise
denoised_signal = denoise_signal(test_signal, 'db4', level=5)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(t, test_signal, 'b-', linewidth=1, alpha=0.7, label='Noisy')
ax.plot(t, denoised_signal, 'r-', linewidth=2, label='Denoised')
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Amplitude', fontsize=11)
ax.set_title('Signal Denoising with Wavelet Thresholding',
             fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('wavelet_denoising.png', dpi=200)
plt.close()

print("Wavelet analysis completed")

## Short-Time Fourier Transform and Spectrograms

### Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq

np.random.seed(42)  # reproducible noise

# Create a chirp signal (frequency increases with time)
t = np.linspace(0, 5, 2000)
# Frequency sweeps from 5 Hz to 15 Hz
f_start, f_end = 5, 15
chirp = signal.chirp(t, f_start, t[-1], f_end, method='linear')

# Add some noise
chirp += 0.3 * np.random.randn(len(t))

# Compute STFT
nperseg = 256  # Window length
noverlap = 128  # Overlap
f_stft, t_stft, Zxx = signal.stft(chirp, fs=len(t)/t[-1],
                                  nperseg=nperseg,
                                  noverlap=noverlap)

# Plot results
fig = plt.figure(figsize=(14, 10))

# Original signal
ax1 = plt.subplot(3, 1, 1)
ax1.plot(t, chirp, 'b-', linewidth=1)
ax1.set_xlabel('Time (s)', fontsize=11)
ax1.set_ylabel('Amplitude', fontsize=11)
ax1.set_title('Input Chirp Signal (5-15 Hz sweep)', fontsize=12)
ax1.grid(True, alpha=0.3)

# Spectrogram
ax2 = plt.subplot(3, 1, 2)
Sxx = np.abs(Zxx)**2
im = ax2.pcolormesh(t_stft, f_stft, 10 * np.log10(Sxx + 1e-10),
                     shading='gouraud', cmap='viridis')
ax2.set_xlabel('Time (s)', fontsize=11)
ax2.set_ylabel('Frequency (Hz)', fontsize=11)
ax2.set_ylim(0, 40)
ax2.set_title('Spectrogram (STFT Magnitude Squared)',
              fontsize=12)
cbar = plt.colorbar(im, ax=ax2)
cbar.set_label('Power (dB)', fontsize=10)

# FFT comparison (no time localization)
ax3 = plt.subplot(3, 1, 3)
freqs = fftfreq(len(chirp), t[1] - t[0])
spectrum = np.abs(fft(chirp))**2
idx = freqs > 0
ax3.semilogy(freqs[idx], spectrum[idx], 'r-', linewidth=1.5)
ax3.set_xlabel('Frequency (Hz)', fontsize=11)
ax3.set_ylabel('Power', fontsize=11)
ax3.set_title('FFT Magnitude Spectrum (No Time Information)',
              fontsize=12)
ax3.set_xlim([0, 30])
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stft_spectrogram.png', dpi=200)
plt.close()

# Compare different window functions.  scipy.signal.stft accepts
# window names directly ('boxcar' is the rectangular window).
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

windows_to_test = [
    ('Hann', 'hann'),
    ('Hamming', 'hamming'),
    ('Blackman', 'blackman'),
    ('Rectangular', 'boxcar')
]

for idx, (label, window_name) in enumerate(windows_to_test):
    ax = axes[idx // 2, idx % 2]

    # Compute STFT with specific window
    f_stft, t_stft, Zxx = signal.stft(chirp, fs=len(t)/t[-1],
                                      nperseg=nperseg,
                                      noverlap=noverlap,
                                      window=window_name)

    Sxx = np.abs(Zxx)**2
    im = ax.pcolormesh(t_stft, f_stft, 10 * np.log10(Sxx + 1e-10),
                       shading='gouraud', cmap='viridis')
    ax.set_xlabel('Time (s)', fontsize=10)
    ax.set_ylabel('Frequency (Hz)', fontsize=10)
    ax.set_ylim(0, 40)
    ax.set_title(f'{label} Window', fontsize=11)
    plt.colorbar(im, ax=ax, label='Power (dB)')

plt.tight_layout()
plt.savefig('stft_windows.png', dpi=200)
plt.close()

print("STFT and spectrogram analysis completed")